# **ColabSeqDisplay: fit a model to your own variant library, in a browser.**

<img src="https://img.shields.io/badge/Paper-not%20yet%20posted-lightgrey" style="max-width: 100%;">
<a href="https://github.com/JasonJiangs/ColabSeqDisplay"><img src="https://img.shields.io/badge/Github-black?logo=github" style="max-width: 100%;"></a>
<a href="https://colab.research.google.com/github/JasonJiangs/ColabSeqDisplay/blob/main/colab/ColabSeqDisplay.ipynb"><img src="https://img.shields.io/badge/Open%20in-Colab-F9AB00?logo=googlecolab&logoColor=white" style="max-width: 100%;"></a>
<a href="https://github.com/JasonJiangs/ColabSeqDisplay"><img src="https://img.shields.io/badge/License-see%20repository-lightgrey" style="max-width: 100%;"></a>

- You measured a **combinatorial variant library** — one row per variant, one column per mutated site, one column per assay condition. This notebook fine-tunes a protein language model on it with LoRA, checks the result against a one-hot floor that any linear model could reach, and hands you something that scores variants you have not made yet.

- **It is for the person who ran the assay.** No code, no YAML, no conda: every choice is a field in the panel below, and the hyperparameters for each (backbone, pooling) pair are read from a registry rather than tuned by you.

- **It asks two real questions and derives the rest.** Which backbone reads your sequences, and which residues it is read out from. Everything that has to happen before training follows from those two answers — a structure-aware backbone needs a wild-type 3Di string, a `cosine_…` pooling needs a pooling region — so preparation is a **step that appears when your answers call for it**, not a second notebook and not a second form. For the default pair nothing needs preparing and that step is not on the page at all.

- **Two files leave here.** `model_bundle.zip` is the trained model — LoRA weights, the head, your library description, the frozen pooling positions and the provenance, a few megabytes. `performance_report.zip` is what it scored: the report table, the figure, and a manifest that says which partition those numbers describe and how many times the test set was read. Download them, keep them, email them. There is no hub, no account and no upload: nothing you load leaves this runtime.

- **Two notebooks.** [ColabSeqDisplay](https://colab.research.google.com/github/JasonJiangs/ColabSeqDisplay/blob/main/colab/ColabSeqDisplay.ipynb) is the whole workflow: your library, the backbone, whatever that choice needs prepared, the training, the report, and the two files you leave with. [ColabSeqDisplay_Predict](https://colab.research.google.com/github/JasonJiangs/ColabSeqDisplay/blob/main/colab/ColabSeqDisplay_Predict.ipynb) scores new variants with a model you trained earlier and needs nothing but the `.zip`. There used to be a third, run once per protein to make a 3Di string and a pooling region; that work is a **step of the main panel** now, it appears only when your answers call for it, and those files no longer have to be downloaded and uploaded back.

- **The science is not ours.** LoRA injection, the training loop, pooling, metrics and splits come from the *SequenceDisplay-Workflow-Optimization* research package (`seqdisplay_opt`), vendored into `colabsd/engine/` so that this notebook installs one repository and runs. Every module there names the file it came from; `ATTRIBUTION.md` collects them.

<font color="red">⚠️ <b>Before you run anything:</b> the run-button cell installs this package from https://github.com/JasonJiangs/ColabSeqDisplay.git, and that URL did not resolve when this notebook was generated. If the clone fails, that is why — put a fork's URL, or the path of a folder you uploaded to this runtime, in the field at the top of that cell.</font>

# How to start

## 1 · Switch this runtime to a GPU

`Runtime` ▸ `Change runtime type` ▸ **T4 GPU** ▸ `Save`. Colab restarts the runtime, which takes a few seconds and clears anything you had already run.

## 2 · Click the run-button

Hover over the cell below and click ▶ on its left. The cell installs the package (1–2 minutes the first time) and then draws the panel that **is** this notebook: every choice you make is a field or a button in it. There is no code to write and no other cell to edit.

**This notebook has two buttons, not one.** The panel reports validation numbers only — including the performance archive it exports, which you can take away without unlocking anything. Reading the locked test partition is the second cell, at the bottom — a separate, counted, deliberate act. That cell explains why.

## 3 · Which GPU

- **T4 — <font color="red">free</font>.** 16 GB, which is enough for most of this. <font color="red">It is also unstable: free sessions are pre-empted, drop their connection, and are capped in length, so a run measured in hours is a run you will probably lose halfway.</font> Mount Drive first — see below — and a dropped session costs you the run in progress rather than everything you have done.
- **L4 (needs Colab Pro).** 24 GB — the smallest card that runs the one backbone the registry says will not fit a T4 (`SaProt-1.3B`) at all, and enough session stability to finish a long run. Slower than an A100.
- **A100 (needs Colab Pro).** 40 GB and much faster. This is the card for a full multi-seed evaluation of a large backbone.
- **No GPU (CPU runtime).** The panel still opens, still loads and checks your library, and still runs the one-hot floor and the report. Anything that has to run the language model itself — training, region discovery, folding a structure — refuses to start and says which runtime it needs, rather than dying halfway through a training loop.

### The free tier will disconnect. Mount Drive before it does.

`Files` — the folder icon in the left margin — then **Mount Drive**, and let Colab run the cell it offers you. Mounting does not stop the disconnect. It gives you one folder, `/content/drive/MyDrive/`, that survives one: everything else in the runtime is thrown away when the session ends, including `/content`, the multi-gigabyte backbone download and anything the panel had written. So mount it before you start something long, and copy what you want to keep into it as it appears — `model_bundle.zip` and `performance_report.zip` above all: a few megabytes between them, and they are the model and the evidence for it. A discovered pooling region is worth copying too — it is the one thing here that costs an hour to make a second time.

### Which backbone fits which card

| backbone | pooled feature | needs a 3Di string | est. min per run on a T4 | where to run it |
|---|---|---|---|---|
| `ESM2-8M` | 320-d | no | 20 | a free T4 |
| `ESM2-35M` | 480-d | no | 40 | a free T4 |
| `ESM2-150M` | 640-d | no | 90 | a free T4, if you mount Drive first |
| `ESM2-650M` | 1280-d | no | 240 | **L4 or A100** — a free session usually drops first |
| `SaProt-35M` | 480-d | **yes** | 45 | a free T4 |
| `SaProt-650M` | 1280-d | **yes** | 260 | **L4 or A100** — a free session usually drops first |
| `SaProt-1.3B` | 1280-d | **yes** | — | **L4 or A100** — will not fit a T4 |

**The minutes are estimates, not measurements.** They come from the backbone registry, which quotes an order-of-magnitude wall-clock figure for **one** training run (one data split, one model seed) over the bundled SlugCas9 example — 16,424 variants of a 1,054-residue protein — on a Colab T4. Twice the rows costs roughly twice the time; a longer protein costs more than proportionally, because attention does. Nobody has timed your card, and no run in this project has ever been timed inside a real Colab runtime; the panel prints the real rate once it is training, and that is the number to trust. The registry's own evaluation protocol is 3 x 3 = 9 runs, so multiply the column by 9 before you plan a full one.

**The last column is this generator's reading of those estimates**, not a measurement either: ≤ 60 min per run is comfortable on a free T4; up to 150 min is fine if your results are on Drive; beyond that a free session is likely to end before the run does. A blank estimate means the registry says the model does not fit a T4's 16 GB at all.

**Hyperparameters are looked up, not tuned by you.** Six of the fourteen (backbone, pooling) pairs on this form carry tuned values, all of them for `cosine_p90_mean` pooling, scoring 0.5486–0.5636 test Spearman on upstream's benchmark:

- `ESM2-35M` · `cosine_p90_mean` — 0.5486
- `ESM2-150M` · `cosine_p90_mean` — 0.5546
- `ESM2-650M` · `cosine_p90_mean` — 0.5636
- `SaProt-35M` · `cosine_p90_mean` — 0.5576
- `SaProt-650M` · `cosine_p90_mean` — 0.5592
- `SaProt-1.3B` · `cosine_p90_mean` — 0.5577

Every other pair on the form is a **placeholder**: every `mutation_site_mean` pair is one, and so is `ESM2-8M` in both poolings. The panel raises a **Warning** the moment you pick one, and every number it prints afterwards is a lower bound rather than a result. Each score above is an average over 3 data splits x 3 model seeds = 9 training runs; the panel defaults to **one** run, which is enough to see whether the machinery works and not enough to quote a ±.

The registry holds ten tuned pairs in all. The other four belong to backbones these notebooks no longer put on the form — *Where the other backbones went*, below, says which and why.

**Notice which pooling that is.** The only pooling anybody has tuned is `cosine_p90_mean`, and that is the pooling that needs a **region** discovered for it. Picking the tuned pair is picking the preparation step below; the panel adds that step, prices it before you press anything, and offers you a region you already have before it offers to compute one.

**Structure.** The backbones marked *needs a 3Di string* read shape as well as sequence, so they need one extra string describing your wild type. **You do not have to go and find it**: picking one of them makes the step that produces it appear below the choice, and picking an ESM2 makes that step disappear again. What it costs is *What step 3 costs*, further down.

### Where the other backbones went

The form offers seven backbones: the ESM2 ladder and SaProt. Seven more are in this package and are **not** on it, across six other families. The two questions in step 2 decide everything below them, so they are kept a real comparison — read the sequence, or read the sequence and the shape — rather than a menu of fourteen. Nothing was deleted:

- `ProtT5-XL` is in this package but the notebooks do not offer it: a 1.2B-parameter encoder that needs an L4 or A100 and an extra sentencepiece install, where the notebooks target a free T4 and install nothing beyond colabsd. Its adapter is still here and still tested — `create_adapter('ProtT5-XL', pooling=...)` builds it from Python, and `config/best/` still carries its hyperparameters.
- `Ankh-large` is in this package but the notebooks do not offer it: a 1.2B-parameter encoder that needs an L4 or A100, where the notebooks target a free T4. Its adapter is still here and still tested — `create_adapter('Ankh-large', pooling=...)` builds it from Python, and `config/best/` still carries its hyperparameters.
- `ESMC-300M` is in this package but the notebooks do not offer it: it loads only through the EvolutionaryScale SDK (`pip install esm`), an install the notebooks do not make on a user's behalf. Its adapter is still here and still tested — `create_adapter('ESMC-300M', pooling=...)` builds it from Python, and `config/best/` still carries its hyperparameters.
- `ESMC-600M` is in this package but the notebooks do not offer it: it loads only through the EvolutionaryScale SDK (`pip install esm`), an install the notebooks do not make on a user's behalf. Its adapter is still here and still tested — `create_adapter('ESMC-600M', pooling=...)` builds it from Python, and `config/best/` still carries its hyperparameters.
- `SeqDance` is in this package but the notebooks do not offer it: a dynamics-pretrained ESM2-35M — a good model, but it answers a narrower question than the sequence-versus-structure choice the notebooks are built around. Its adapter is still here and still tested — `create_adapter('SeqDance', pooling=...)` builds it from Python, and `config/best/` still carries its hyperparameters.
- `ESMDance` is in this package but the notebooks do not offer it: a dynamics-tuned ESM2-35M whose pooled feature is its 50-dim prediction head rather than the trunk, so it is not read like the other entries in a single comparison. Its adapter is still here and still tested — `create_adapter('ESMDance', pooling=...)` builds it from Python, and `config/best/` still carries its hyperparameters.
- `METL` is in this package but the notebooks do not offer it: Rosetta-pretrained and protein-specific, with no HuggingFace weights for anything to load. It has no adapter in this package, so there is nothing here to run it with.

Putting a family back on the form is one line: its name moves out of `WITHHELD_FAMILY_REASONS` and into `OFFERED_FAMILIES` in `colabsd/backbones/registry.py` — plus a tuned entry in `config/best/` if you want the panel to stop calling the result provisional. And a `model_bundle.zip` trained on one of them **still scores**: the Predict notebook reads the bundle, not this list, and says in words why the backbone it names is no longer offered.

### Step 3 is derived from step 2, and is often not there at all

You answer two questions in the panel — which backbone, which pooling — and it works out what has to exist before training rather than asking you a third time:

| what you answered in step 2 | what step 3 then asks for |
|---|---|
| a **SaProt** backbone | a wild-type **3Di string** — the structural alphabet that backbone reads beside each residue |
| **`cosine_p90_mean`** pooling | a **pooling region** — the residue positions the embedding is averaged over |
| **ESM2** with **`mutation_site_mean`** | nothing at all — there is no step 3 on the page |

A control that has nothing to do with what you picked is not greyed out, it is **gone**: choose an ESM2 and every 3Di field leaves the page. Change your mind in step 2 and step 3 changes with it, or vanishes — and the steps after it renumber, because a page that skips from 2 to 4 reads as a step you failed to find. When both jobs are needed they are lettered **3a** and **3b**; a single job just sits under the step's own heading.

**Nothing is downloaded and uploaded back.** Both artefacts stay in this session — written into the work folder, which is why mounting Drive is worth the thirty seconds — and when one already exists the step says so and offers to **reuse** it rather than compute it again. The bundled SlugCas9 example ships a 3Di string and both region files, so it runs end to end with nothing to prepare and nothing to press.

### What step 3 costs

**The 3Di string is nearly free — if you have a structure.** Download the AlphaFold model for your protein from [alphafold.ebi.ac.uk](https://alphafold.ebi.ac.uk), upload the `.cif`, and the conversion takes seconds on any runtime, GPU or not. Only `SaProt-35M`, `SaProt-650M`, `SaProt-1.3B` need it at all.

**Folding it yourself is the last resort.** <font color="red">ESMFold builds one value per residue *pair*, so its memory grows with the square of the length. `colabsd.structure` calls **700 residues** the most a free 16 GB T4 can be expected to fold and warns you past it, and the panel will not fold anything at all until you tick a box saying you accept the memory risk.</font> An AlphaFold structure is faster, free and more accurate. Try that first.

**Region discovery is the expensive step.** It pushes full-length variant sequences through `facebook/esm2_t33_650M_UR50D` and scores how much each residue's embedding disagrees across your library — no gradients, but one forward pass per variant.

<font color="red">**Region discovery and a fine-tune do not both fit in one free session.**</font> Discovering a region over all 16,424 rows of the bundled example is 27 min to 2 h 39 min on a GPU, and the fine-tune that follows is another ~40 minutes for the default `ESM2-35M` — more for anything bigger. The slow end of that discovery, on its own, is past the two hours this package treats as one free session — before a single training step has run. Three ways out, in the order the panel offers them:

1. **Reuse a region instead of discovering one.** The step lists what is already here — the bundled example's own region files, and anything this session has made — and reuse is the default whenever there is something to reuse. Region discovery is the one job in this notebook you do not want to pay for twice.
2. **Score a subsample.** The field starts at **2,000 variants**, which is 3 min to 19 min rather than 27 min to 2 h 39 min, and the panel shows you the exact rows it would score and that estimate *before* the button.
3. **Mount Drive first** (above). The region file is written into the work folder, so a session that drops after discovery and before training costs you the training run and not the region.

The panel will also **not start discovery on your behalf**. Pressing **Train** with a region still undiscovered stops and says so, rather than quietly beginning an hour of GPU work you did not ask for.

Both outputs are deterministic given their inputs, so a dropped session costs you time and nothing else — as long as what had already been written was written somewhere that survives.

### The two files you leave with

- **`model_bundle.zip`** — **the model.** The LoRA weights, the head, your library description, the *frozen* pooling coordinates, the hyperparameters and the provenance: a few megabytes. This is the file the Predict notebook asks for, and scoring from it never needs a region file or a 3Di string again.
- **`performance_report.zip`** — **the numbers.** `report.csv`, `report.png`, `report.json`, plus a `performance.json` and a `README.txt` that state **which partition those numbers describe** and **how many times the test set has been read**. Open it in six months and it still answers both questions without you having to remember.

**You do not have to unlock anything to get `performance_report.zip`.** Press its button with the test partition still locked and the archive comes back full of **validation** numbers and says so — which is the state you should be in while you are still deciding anything. Unlock the test set in the last cell of this notebook and the same archive is rewritten, now carrying the test numbers and the unlock count.

In [ ]:
#@title **Click the run-button to use ColabSeqDisplay** { display-mode: "form" }

#@markdown ### Hint
#@markdown - The first run of this cell installs ColabSeqDisplay: **1-2 minutes** on a fresh runtime, with nothing for you to do while it works. Run it again later in the same session and it skips straight to the panel.
#@markdown - **The panel this cell draws below itself is the whole program.** Answer what it asks, press the buttons it offers. There is no code to write and nothing else in this notebook to edit.
#@markdown - **A file picker freezes this page while it is open.** Anything that asks you for a file — your library, a structure, a saved model — opens Colab's own upload dialog, and that dialog blocks the notebook: until you pick a file or cancel, every control is frozen and nothing on screen will change. The panel names the file it is waiting for *before* the dialog opens, and **Cancel upload** in the dialog gives the page back.
#@markdown - **What the run-button is telling you.** The ▶ arrow means nothing is running: click it to start. It spins while the cell installs the package and builds the panel, then goes back to ▶ — that means finished, not broken. The panel stays live after the cell ends, for as long as this runtime does; if it ever stops responding, click ▶ again to rebuild it.
#@markdown ### <font color=red>If the session disconnects</font>
#@markdown - <font color=red>Colab drops long sessions, and the free T4 drops them soonest. Reconnect, run this cell again, and the panel comes back — but the runtime is empty: whatever was under `/content` is gone, whatever you wrote to a mounted Google Drive folder is not. Mount Drive before you start anything long.</font>
#@markdown - <font color=red>Changing the runtime type restarts Python and empties it just the same. Stop this cell first, change the runtime, then run it again.</font>
#@markdown ### Where the code comes from
#@markdown - The field below names what gets installed — **one repository, and it is the whole program**: the fine-tuning engine ships inside it. A **folder path** works as well as a URL — the path of a checkout you uploaded to this runtime — and is installed with `pip install -e`, which keeps its `config/best/` registry and its bundled `examples/` where you can read and edit them.
#@markdown - <font color=red>As generated, this field names a repository that was not published yet. If the clone fails, that is why, and the error names the field to change.</font>
colabsd_repository = "https://github.com/JasonJiangs/ColabSeqDisplay.git"  #@param {type:"string"}

import importlib
import importlib.util
import subprocess
import sys
from pathlib import Path

WORK_ROOT = Path.cwd()


def run_command(command):
    """Run a command, raising with its own output when it fails."""
    parts = [str(part) for part in command]
    finished = subprocess.run(parts, capture_output=True, text=True)
    if finished.returncode != 0:
        raise RuntimeError(
            "This command failed:\n  " + " ".join(parts) + "\n"
            + (finished.stdout or "")[-1500:] + (finished.stderr or "")[-1500:]
        )
    return finished


def checkout(source, name):
    """A local folder as given, or a shallow clone of a git URL beside this notebook."""
    local = Path(source).expanduser()
    if local.is_dir():
        return local.resolve()
    target = WORK_ROOT / name
    if not (target / ".git").is_dir():
        print("cloning " + str(source) + " ...")
        try:
            run_command(["git", "clone", "--depth", "1", source, target])
        except RuntimeError as exc:
            raise RuntimeError(
                str(exc) + "\n\n" + name + " could not be downloaded from " + str(source) + ". Put a "
                "repository this runtime can reach in the field at the top of this form, or the path of a "
                "folder you uploaded to this runtime (for example " + str(target) + ")."
            ) from None
    return target.resolve()


def package_dir(module):
    """The directory an importable package sits in, or None when it is not importable."""
    found = importlib.util.find_spec(module)
    if found is None or not found.origin:
        return None
    return Path(found.origin).resolve().parent


def colabsd_is_complete():
    """True when colabsd is importable *and* its config registry and bundled example came with it.

    Two layouts are both correct: an editable install leaves `config/` and `examples/` beside the
    package, a built wheel carries them inside it. Either answer counts; neither does.
    """
    package = package_dir("colabsd")
    if package is None:
        return False
    return any(
        (root / "config" / "best").is_dir() and (root / "examples").is_dir()
        for root in (package, package.parent)
    )


if not colabsd_is_complete():
    package_root = checkout(colabsd_repository, "ColabSeqDisplay")
    print("installing ColabSeqDisplay from " + str(package_root) + " ...")
    run_command([sys.executable, "-m", "pip", "install", "-q", "-e", package_root])
    importlib.invalidate_caches()
    if str(package_root) not in sys.path:
        sys.path.insert(0, str(package_root))

import colabsd
from colabsd.ui import core, main_workflow

runtime = core.detect_runtime()
WORK_DIR = WORK_ROOT / "colabsd_work"

if runtime.has_gpu:
    GPU_DESCRIPTION = str(runtime.gpu_name) + "  (" + format(runtime.gpu_memory_gb or 0.0, ".1f") + " GB)"
else:
    GPU_DESCRIPTION = "none — Runtime > Change runtime type > T4 GPU, then run this cell again"

print("colabsd " + colabsd.__version__ + "   from " + str(Path(colabsd.REPO_ROOT)))
print("GPU       " + GPU_DESCRIPTION)
print("files     " + str(WORK_DIR))
print("")

wizard = main_workflow.launch(work_dir=WORK_DIR)

In [ ]:
#@title **Click the run-button to unlock the test set and write the final report** { display-mode: "form" }

#@markdown ### Why this is a separate button
#@markdown - Everything the panel above reports is measured on the **validation** split, on purpose. You chose the backbone, the pooling and the number of runs by looking at those numbers, so they are no longer an honest estimate of how the model behaves on data nobody has looked at.
#@markdown - The **test** partition stays locked while all of that happens, and is read here: once, deliberately, by you. That is the whole point of holding it back — a number you consult while you are still making decisions stops being a test number.
#@markdown - **Every unlock is counted.** The count is written to `unlock.json` beside the run and printed in the report, so whoever reads the result can see how many times the test set was opened. Unlock once, at the end, when you have stopped changing things.
#@markdown - Nothing here retrains anything. If you unlock, then change something and train again, you unlock again — the count goes up, and the report says so.
#@markdown ### What this cell writes
#@markdown - **You did not need this cell to read your results.** The panel above writes `performance_report.zip` with the **validation** numbers in it while the test partition is still locked, and that is the archive to read while you are still deciding anything.
#@markdown - This cell rewrites that same `performance_report.zip` — the report table, the figure and the JSON that records what they show — so that it now carries the **test** numbers and the unlock count, and offers it to you along with the figure on its own. Whatever you keep, its own contents say which partition it describes and how many times the test set had been opened when it was written.

try:
    from colabsd.ui import main_workflow, unlock
except ImportError:
    raise RuntimeError(
        "ColabSeqDisplay is not installed in this runtime yet. Run the cell above first: it installs the "
        "package and trains the model this cell reports on."
    ) from None

try:
    trained = wizard
except NameError:
    raise RuntimeError(
        "Run the cell above first, and train something in the panel it draws. This cell reports on that "
        "run: it reads the panel the cell above leaves behind as `wizard`, and without it "
        "there is nothing to unlock."
    ) from None

unlock.launch(
    trained,
    output_dir=main_workflow.run_dir(trained.state),
    work_dir=main_workflow.work_dir(trained.state),
)